# 第2章: 個票データによる離散選択モデルの推定（Python版）

RコードをPythonに変換したJupyterノートブック版です。

**内容:**
1. データの読み込みとクリーニング
2. 記述統計
3. 分析用データの整形
4. スクラッチでの推定（多項ロジット・ランダム係数ロジット × 属性なし・あり）
5. 推定結果のまとめ
6. 選好パラメータ・WTPの分布
7. 需要曲線・収入曲線

## 1. セットアップ

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from scipy.optimize import minimize
from scipy.linalg import inv
from pathlib import Path
import warnings
import time

warnings.filterwarnings('ignore')

# 日本語フォント設定
for font in ['Hiragino Sans', 'Hiragino Kaku Gothic Pro', 'Yu Gothic', 'IPAexGothic']:
    try:
        matplotlib.rcParams['font.family'] = font
        break
    except:
        continue
matplotlib.rcParams['axes.unicode_minus'] = False

# パスの設定（python/サブディレクトリから実行する前提）
BASE_DIR = Path('..')
DATA_DIR = BASE_DIR / 'data'
OUTPUT_DIR = BASE_DIR / 'output'
OUTPUT_DIR.mkdir(exist_ok=True)

print("セットアップ完了")


## 2. データの読み込みとクリーニング

In [ ]:
# データの読み込み
KT_2024 = pd.read_csv(DATA_DIR / "KinokoTakenokoSurvey_raw.csv", encoding='utf-8')

# 列名変更（Rの select(ID, c(8:17)) に対応 → 0-indexedで列7〜16）
new_col_name = ["experience", "Q1", "Q2", "Q3", "Q4", "Q5",
                "age", "gender", "region", "familyhouse"]

KT_2024 = KT_2024.iloc[:, 7:17].copy()
KT_2024.columns = new_col_name

# IDを追加（Rの row_number() に対応）
KT_2024.insert(0, 'ID', range(1, len(KT_2024) + 1))

# データクリーニング: 食べたことがない・回答したくないを除外し、NAを除外
KT_2024 = KT_2024[
    (KT_2024['experience'] != "4 : 食べたことがない") &
    (KT_2024['gender'] != "3 : 回答したくない")
].dropna().reset_index(drop=True)

# IDを振り直す
KT_2024['ID'] = range(1, len(KT_2024) + 1)

N = len(KT_2024)
print(f"回答者数 N = {N}")
KT_2024.head()


## 3. 記述統計

### 3.1 回答数の比率

In [ ]:
# 回答数の比率の表を作成
datafig = KT_2024[['ID', 'Q1', 'Q2', 'Q3', 'Q4', 'Q5']].melt(
    id_vars=['ID'], var_name='Q', value_name='choice')

# 対応する選択肢に置き換える
Q_mapping = {
    'Q1': 'Q1: (200円, 200円)',
    'Q2': 'Q2: (180円, 200円)',
    'Q3': 'Q3: (200円, 170円)',
    'Q4': 'Q4: (220円, 200円)',
    'Q5': 'Q5: (190円, 210円)'
}
datafig['Q'] = datafig['Q'].map(Q_mapping)

tab_choice = (datafig.groupby(['Q', 'choice']).size().reset_index(name='n')
              .assign(n=lambda x: x['n'] / N)
              .pivot(index='Q', columns='choice', values='n')
              .reset_index())

tab_choice.to_csv(OUTPUT_DIR / "tab2_1_tab_choice.txt", sep='\t', index=False)
print("tab2_1_tab_choice.txt を出力しました")
tab_choice


### 3.2 回答者の属性

In [ ]:
# 回答者の属性の比率
data_atr = KT_2024.copy()
data_atr['gender_j'] = data_atr['gender'].map({"1 : 男性": "男性", "2 : 女性": "女性"})
data_atr['exp_j'] = data_atr['experience'].map({
    "1 : 過去半年以内": "過去半年以内",
    "2 : 過去半年から１年以内": "過去半年から１年以内",
    "3 : １年以上前": "１年以上前"
})
data_atr['adult_j'] = (data_atr['age'] >= 20).map({True: "成人", False: "未成年"})
data_atr['region_j'] = data_atr['region'].map({
    "1 : 北海道地方": "関東", "2 : 東北地方": "関東",
    "3 : 関東地方": "関東", "4 : 中部地方": "関東",
    "5 : 近畿地方": "関西", "6 : 中国地方": "関西",
    "7 : 四国地方": "関西", "8 : 九州地方(沖縄含む)": "関西",
    "9 : 海外": "海外"
})
data_atr['fam_j'] = data_atr['familyhouse'].map({1: "実家暮らし", 0: "実家暮らしでない"})

tab_atr_list = []
for var_name, col in [("性別", "gender_j"), ("成人かどうか", "adult_j"),
                       ("実家暮らしかどうか", "fam_j"), ("出身地方", "region_j")]:
    t = data_atr.groupby(col).size().reset_index(name='n')
    t['割合'] = t['n'] / N
    t['変数'] = var_name
    t.rename(columns={col: '属性'}, inplace=True)
    tab_atr_list.append(t[['変数', '属性', '割合']])

tab_atr = pd.concat(tab_atr_list, ignore_index=True)
tab_atr


## 4. 分析用データの整形

In [ ]:
# 分析用のデータ整形（wide → long）
data_for_estimation = KT_2024[['ID', 'Q1', 'Q2', 'Q3', 'Q4', 'Q5',
                                'gender', 'experience', 'age', 'region', 'familyhouse']].copy()
data_for_estimation = data_for_estimation.melt(
    id_vars=['ID', 'gender', 'experience', 'age', 'region', 'familyhouse'],
    value_vars=['Q1', 'Q2', 'Q3', 'Q4', 'Q5'],
    var_name='occasion', value_name='choice')

# 提示される価格の組み合わせ
pricedata = pd.DataFrame({
    'occasion': ['Q1', 'Q2', 'Q3', 'Q4', 'Q5'],
    'price_0': [0, 0, 0, 0, 0],
    'price_1': [200, 180, 200, 220, 190],
    'price_2': [200, 200, 170, 200, 210]
})
data_for_estimation = data_for_estimation.merge(pricedata, on='occasion', how='left')

# ダミー変数の作成
data_for_estimation['Kinoko_0'] = 0
data_for_estimation['Kinoko_1'] = 1
data_for_estimation['Kinoko_2'] = 0
data_for_estimation['Takenoko_0'] = 0
data_for_estimation['Takenoko_1'] = 0
data_for_estimation['Takenoko_2'] = 1

data_for_estimation = data_for_estimation.sort_values(['ID', 'occasion']).reset_index(drop=True)

# 選択肢の数値化
data_for_estimation['choice'] = data_for_estimation['choice'].map({
    "1 : きのこの山を買う": 1,
    "2 : たけのこの里を買う": 2,
    "3 : どちらも買わない": 0
})

# 属性の変換（分析用）
data_for_estimation['gender'] = data_for_estimation['gender'].map({
    "1 : 男性": "0", "2 : 女性": "female"})
data_for_estimation['exp'] = data_for_estimation['experience'].map({
    "1 : 過去半年以内": "halfyear",
    "2 : 過去半年から１年以内": "half_to_1year",
    "3 : １年以上前": "0"})
data_for_estimation['adult'] = (data_for_estimation['age'] >= 20).astype(int)
data_for_estimation['region'] = data_for_estimation['region'].map({
    "1 : 北海道地方": "_Kanto", "2 : 東北地方": "_Kanto",
    "3 : 関東地方": "_Kanto", "4 : 中部地方": "_Kanto",
    "5 : 近畿地方": "Kansai", "6 : 中国地方": "Kansai",
    "7 : 四国地方": "Kansai", "8 : 九州地方(沖縄含む)": "Kansai",
    "9 : 海外": "Oversea"})
data_for_estimation['choiceid'] = range(1, len(data_for_estimation) + 1)

print(f"分析用データの行数: {len(data_for_estimation)}")
print(f"回答者数: {len(data_for_estimation['ID'].unique())}")
data_for_estimation.head(10)


## 5. 関数定義

高速化のためNumPyベクトル演算を活用した尤度関数を定義します。

In [ ]:
from scipy.special import logsumexp

# 価格の配列（Q1〜Q5に対応）
PRICES_KINOKO = np.array([200, 180, 200, 220, 190], dtype=np.float64)
PRICES_TAKENOKO = np.array([200, 200, 170, 200, 210], dtype=np.float64)


# === 数値安定な対数選択確率関数 ===

def log_choice_probs(util_k, util_t):
    """
    数値安定なlog-sum-exp方式で対数選択確率を計算
    util_k, util_t: 任意形状（スカラー、1D、2D対応）
    returns: (log_prob_k, log_prob_t, log_prob_o)
    """
    util_o = np.zeros_like(util_k)  # 外部財の効用 = 0
    stacked = np.stack([util_o, util_k, util_t], axis=-1)
    log_denom = logsumexp(stacked, axis=-1)
    log_prob_k = util_k - log_denom
    log_prob_t = util_t - log_denom
    log_prob_o = -log_denom  # log(1) - log_denom = -log_denom
    return log_prob_k, log_prob_t, log_prob_o


# === 選択確率関数 ===

def f_logit_prob(alpha_kinoko, alpha_takenoko, beta, price_kinoko, price_takenoko):
    """消費者属性なし多項ロジットの選択確率"""
    util_k = alpha_kinoko - beta * price_kinoko
    util_t = alpha_takenoko - beta * price_takenoko
    lp_k, lp_t, lp_o = log_choice_probs(util_k, util_t)
    return np.array([np.exp(lp_k), np.exp(lp_t), np.exp(lp_o)])


# === 需要関数 ===

def f_demand(alpha_kinoko_vec, alpha_takenoko_vec, beta_vec, price_kinoko, price_takenoko):
    """各個人の選択確率を足し合わせて需要を得る（ベクトル化版）"""
    util_k = alpha_kinoko_vec - beta_vec * price_kinoko
    util_t = alpha_takenoko_vec - beta_vec * price_takenoko
    lp_k, lp_t, lp_o = log_choice_probs(util_k, util_t)
    prob_k = np.exp(lp_k)
    prob_t = np.exp(lp_t)
    prob_o = np.exp(lp_o)
    return np.array([np.sum(prob_k), np.sum(prob_t), np.sum(prob_o)])


# === 対数尤度関数（属性なし多項ロジット）===

def f_likelihood_logit(param, respondent_choices):
    """
    属性なし多項ロジットの対数尤度関数（ベクトル化版）
    param: [alpha_kinoko, alpha_takenoko, beta]
    respondent_choices: (N, 5) array
    """
    alpha_k, alpha_t, beta = param[0], param[1], param[2]
    util_k = alpha_k - beta * PRICES_KINOKO  # (5,)
    util_t = alpha_t - beta * PRICES_TAKENOKO  # (5,)
    lp_k, lp_t, lp_o = log_choice_probs(util_k, util_t)

    log_cp = np.where(respondent_choices == 0, lp_o[np.newaxis, :],
             np.where(respondent_choices == 1, lp_k[np.newaxis, :],
                      lp_t[np.newaxis, :]))
    return np.sum(log_cp)


# === 対数尤度関数（属性あり多項ロジット）===

def f_likelihood_logit_with_atr(param, respondent_choices, respondent_atrs):
    """
    属性あり多項ロジットの対数尤度関数（ベクトル化版）
    param: 18要素
    """
    beta = param[0]
    alpha_k = param[1]
    alpha_t = param[2]
    param_atr_price = param[[3, 6, 9, 12, 13]]
    param_atr_takenoko = param[[4, 7, 10, 14, 15]]
    param_atr_kinoko = param[[5, 8, 11, 16, 17]]

    # 属性交差項: (N,)
    atr_k = respondent_atrs @ param_atr_kinoko
    atr_t = respondent_atrs @ param_atr_takenoko
    atr_p = respondent_atrs @ param_atr_price

    # 効用: (N, 5)
    util_k = (alpha_k + atr_k[:, None]) - PRICES_KINOKO[None, :] * (beta + atr_p[:, None])
    util_t = (alpha_t + atr_t[:, None]) - PRICES_TAKENOKO[None, :] * (beta + atr_p[:, None])

    lp_k, lp_t, lp_o = log_choice_probs(util_k, util_t)

    log_cp = np.where(respondent_choices == 0, lp_o,
             np.where(respondent_choices == 1, lp_k, lp_t))
    return np.sum(log_cp)


# === 対数尤度関数（属性なしランダム係数ロジット）===

def f_likelihood_rcdclogit(param, respondent_choices, tau, times_draw, n_resp):
    """
    属性なしランダム係数ロジットの対数尤度（log-sum-exp安定版）
    param: [beta, alpha_kinoko, alpha_takenoko, sigma_price, sigma_kinoko, sigma_takenoko]
    """
    beta = param[0]
    alpha_k, alpha_t = param[1], param[2]
    sigma_p, sigma_k, sigma_t = param[3], param[4], param[5]

    total_ll = 0.0
    for i in range(n_resp):
        tau_i = tau[times_draw * i : times_draw * (i + 1), :]  # (D, 3)

        ak = alpha_k + sigma_k * tau_i[:, 0]  # (D,)
        at = alpha_t + sigma_t * tau_i[:, 1]
        bd = beta + sigma_p * tau_i[:, 2]

        # 効用: (D, 5)
        util_k = ak[:, None] - bd[:, None] * PRICES_KINOKO[None, :]
        util_t = at[:, None] - bd[:, None] * PRICES_TAKENOKO[None, :]

        lp_k, lp_t, lp_o = log_choice_probs(util_k, util_t)

        choices_i = respondent_choices[i]  # (5,)
        log_cp = np.where(choices_i[None, :] == 0, lp_o,
                 np.where(choices_i[None, :] == 1, lp_k, lp_t))  # (D, 5)

        # ドローごとの5問の対数同時確率
        log_joint = np.sum(log_cp, axis=1)  # (D,)
        # log(mean(exp(log_joint))) = logsumexp(log_joint) - log(D)
        log_mean_joint = logsumexp(log_joint) - np.log(times_draw)
        total_ll += log_mean_joint

    return total_ll


# === 対数尤度関数（属性ありランダム係数ロジット）===

def f_likelihood_rcdclogit_with_atr(param, respondent_choices, respondent_atrs,
                                     tau, times_draw, n_resp):
    """
    属性ありランダム係数ロジットの対数尤度（log-sum-exp安定版）
    param: 21要素
    """
    beta = param[0]
    alpha_k, alpha_t = param[1], param[2]
    sigma_p, sigma_k, sigma_t = param[3], param[4], param[5]
    param_atr_price = param[[6, 9, 12, 15, 16]]
    param_atr_takenoko = param[[7, 10, 13, 17, 18]]
    param_atr_kinoko = param[[8, 11, 14, 19, 20]]

    total_ll = 0.0
    for i in range(n_resp):
        tau_i = tau[times_draw * i : times_draw * (i + 1), :]
        atr_i = respondent_atrs[i]

        ak_draw = alpha_k + sigma_k * tau_i[:, 0]
        at_draw = alpha_t + sigma_t * tau_i[:, 1]
        bd_draw = beta + sigma_p * tau_i[:, 2]

        atr_k_val = np.dot(param_atr_kinoko, atr_i)
        atr_t_val = np.dot(param_atr_takenoko, atr_i)
        atr_p_val = np.dot(param_atr_price, atr_i)

        util_k = (ak_draw[:, None] + atr_k_val) - PRICES_KINOKO[None, :] * (bd_draw[:, None] + atr_p_val)
        util_t = (at_draw[:, None] + atr_t_val) - PRICES_TAKENOKO[None, :] * (bd_draw[:, None] + atr_p_val)

        lp_k, lp_t, lp_o = log_choice_probs(util_k, util_t)

        choices_i = respondent_choices[i]
        log_cp = np.where(choices_i[None, :] == 0, lp_o,
                 np.where(choices_i[None, :] == 1, lp_k, lp_t))

        log_joint = np.sum(log_cp, axis=1)
        log_mean_joint = logsumexp(log_joint) - np.log(times_draw)
        total_ll += log_mean_joint

    return total_ll


# === 標準誤差の計算 ===

def compute_se(neg_ll_func, param_est):
    """数値微分によるヘシアンから標準誤差を計算"""
    n = len(param_est)
    eps = np.maximum(1e-5, np.abs(param_est) * 1e-4)
    hess = np.zeros((n, n))
    f0 = neg_ll_func(param_est)
    if not np.isfinite(f0):
        return None

    for i in range(n):
        x_p = param_est.copy(); x_p[i] += eps[i]
        x_m = param_est.copy(); x_m[i] -= eps[i]
        fp = neg_ll_func(x_p)
        fm = neg_ll_func(x_m)
        if not (np.isfinite(fp) and np.isfinite(fm)):
            return None
        hess[i, i] = (fp - 2 * f0 + fm) / (eps[i] ** 2)
        for j in range(i + 1, n):
            x_pp = param_est.copy(); x_pp[i] += eps[i]; x_pp[j] += eps[j]
            x_pm = param_est.copy(); x_pm[i] += eps[i]; x_pm[j] -= eps[j]
            x_mp = param_est.copy(); x_mp[i] -= eps[i]; x_mp[j] += eps[j]
            x_mm = param_est.copy(); x_mm[i] -= eps[i]; x_mm[j] -= eps[j]
            hess[i, j] = (neg_ll_func(x_pp) - neg_ll_func(x_pm) - neg_ll_func(x_mp) + neg_ll_func(x_mm)) / (4 * eps[i] * eps[j])
            hess[j, i] = hess[i, j]

    hess = (hess + hess.T) / 2
    try:
        cov = inv(hess)
        se = np.sqrt(np.diag(cov))
        if np.any(np.isnan(se)) or np.any(se <= 0):
            return None
        return np.round(se, 4)
    except:
        return None

print("関数定義完了")


## 6. 推定用データの準備

In [ ]:
# 回答者ごとの選択データを(N, 5)の配列に変換
unique_ids = sorted(data_for_estimation['ID'].unique())
n_respondents = len(unique_ids)

# 各回答者のQ1〜Q5の選択を配列化
respondent_choices = np.zeros((n_respondents, 5), dtype=int)
for idx, uid in enumerate(unique_ids):
    mask = data_for_estimation['ID'] == uid
    row = data_for_estimation[mask].sort_values('occasion')
    respondent_choices[idx, :] = row['choice'].values

print(f"回答者数: {n_respondents}")
print(f"respondent_choices shape: {respondent_choices.shape}")
print(f"選択の分布: 0={np.sum(respondent_choices==0)}, 1={np.sum(respondent_choices==1)}, 2={np.sum(respondent_choices==2)}")


## 7. モデル1: 消費者属性を入れない多項ロジット（スクラッチ推定）

Rの`optimx(method="Nelder-Mead")`に対応します。

In [ ]:
# 初期パラメータ [alpha_kinoko, alpha_takenoko, beta]
ini_ml = np.array([5.0, 5.0, -0.01])

# 負の対数尤度関数
def neg_ll_ml(param):
    return -f_likelihood_logit(param, respondent_choices)

start_time = time.time()
result_ml = minimize(neg_ll_ml, ini_ml, method='Nelder-Mead',
                     options={'xatol': 1e-8, 'fatol': 1e-8, 'maxiter': 10000})
elapsed_ml = time.time() - start_time

est_ml = result_ml.x
ll_ml = -result_ml.fun

# 標準誤差の計算
se_ml = compute_se(neg_ll_ml, est_ml)

# Rと同じ並びで表示: [price, Kinoko, Takenoko]
print("=" * 60)
print("モデル1: 消費者属性なし多項ロジット")
print("=" * 60)
print(f"対数尤度: {ll_ml:.6f}")
print(f"計算時間: {elapsed_ml:.2f}秒")
print(f"収束: {'成功' if result_ml.success else '失敗'}")
print()
coef_names_ml = ['price', 'Kinoko', 'Takenoko']
est_ml_reord = est_ml[[2, 0, 1]]
se_ml_reord = se_ml[[2, 0, 1]] if se_ml is not None else np.array([np.nan]*3)

for name, est, se in zip(coef_names_ml, est_ml_reord, se_ml_reord):
    print(f"  {name:12s}: est={est:10.6f}, se={se:.4f}")


## 8. モデル2: 消費者属性を入れた多項ロジット（スクラッチ推定）

Rの`optimx(method="BFGS")`に対応します。

In [ ]:
# 属性ダミーを作成
data_est_atr = data_for_estimation.copy()
data_est_atr['gender_d'] = (data_est_atr['gender'] == 'female').astype(int)
data_est_atr['kansai'] = (data_est_atr['region'] == 'Kansai').astype(int)
data_est_atr['oversea'] = (data_est_atr['region'] == 'Oversea').astype(int)

# 回答者ごとの属性を(N, 5)の配列に変換: [gender, familyhouse, adult, kansai, oversea]
respondent_atrs = np.zeros((n_respondents, 5))
for idx, uid in enumerate(unique_ids):
    row = data_est_atr[data_est_atr['ID'] == uid].iloc[0]
    respondent_atrs[idx, :] = [row['gender_d'], row['familyhouse'],
                                row['adult'], row['kansai'], row['oversea']]

print(f"属性データ shape: {respondent_atrs.shape}")
print(f"女性比率: {respondent_atrs[:, 0].mean():.3f}")
print(f"実家暮らし比率: {respondent_atrs[:, 1].mean():.3f}")
print(f"成人比率: {respondent_atrs[:, 2].mean():.3f}")
print(f"関西比率: {respondent_atrs[:, 3].mean():.3f}")
print(f"海外比率: {respondent_atrs[:, 4].mean():.3f}")


In [ ]:
# 初期値（Rと同じ）
ini_ml_atr = np.array([0, 10, 10, 0, 1, 1, 0, 0, 0, 0, -1, -1, 0, 0, 1, 1, 1, 1], dtype=float)

def neg_ll_ml_atr(param):
    return -f_likelihood_logit_with_atr(param, respondent_choices, respondent_atrs)

start_time = time.time()
result_ml_atr = minimize(neg_ll_ml_atr, ini_ml_atr, method='BFGS',
                          options={'maxiter': 10000, 'gtol': 1e-6})
elapsed_ml_atr = time.time() - start_time

est_ml_atr = result_ml_atr.x
ll_ml_atr = -result_ml_atr.fun

# 標準誤差
se_ml_atr = compute_se(neg_ll_ml_atr, est_ml_atr)

print("=" * 60)
print("モデル2: 消費者属性あり多項ロジット")
print("=" * 60)
print(f"対数尤度: {ll_ml_atr:.6f}")
print(f"計算時間: {elapsed_ml_atr:.2f}秒")
print(f"収束: {'成功' if result_ml_atr.success else '失敗'}")
print()

# Rと同じ係数名
coef_names_ml_atr = [
    'price', 'Kinoko', 'Takenoko',
    'price:genderfemale', 'Takenoko:genderfemale', 'Kinoko:genderfemale',
    'price:familyhouse', 'Takenoko:familyhouse', 'Kinoko:familyhouse',
    'price:adult', 'Takenoko:adult', 'Kinoko:adult',
    'price:regionKansai', 'price:regionOversea',
    'Takenoko:regionKansai', 'Takenoko:regionOversea',
    'Kinoko:regionKansai', 'Kinoko:regionOversea'
]

for i, name in enumerate(coef_names_ml_atr):
    se_val = se_ml_atr[i] if se_ml_atr is not None else np.nan
    print(f"  {name:28s}: est={est_ml_atr[i]:10.6f}, se={se_val:.4f}")


## 9. モデル3: 消費者属性を入れないランダム係数ロジット（スクラッチ推定）

Rの`optimx(method="BFGS")`に対応します。1000回のシミュレーションドローを使用します。

**注意**: この推定には数分かかることがあります。

In [ ]:
# 事前にドローを用意する（Rと同じ乱数シード）
times_draw = 1000
tau = np.zeros((times_draw * n_respondents, 3))

for i in range(n_respondents):
    rng = np.random.RandomState(500 + i + 1)  # Rの set.seed(500+i) に対応
    tau[times_draw * i : times_draw * (i + 1), 0] = rng.randn(times_draw)
    tau[times_draw * i : times_draw * (i + 1), 1] = rng.randn(times_draw)
    tau[times_draw * i : times_draw * (i + 1), 2] = rng.randn(times_draw)

print(f"tau shape: {tau.shape}")
print(f"ドロー回数: {times_draw}")


In [ ]:
# 初期値（Rと同じ）
ini_rcdc = np.array([0.6, 20, 20, 0.1, 1, 1], dtype=float)

def neg_ll_rcdc(param):
    return -f_likelihood_rcdclogit(param, respondent_choices, tau, times_draw, n_respondents)

print("ランダム係数ロジット（属性なし）の推定を開始...")
start_time = time.time()
result_rcdc = minimize(neg_ll_rcdc, ini_rcdc, method='BFGS',
                       options={'maxiter': 10000, 'gtol': 1e-5})
elapsed_rcdc = time.time() - start_time

est_rcdc = result_rcdc.x
ll_rcdc = -result_rcdc.fun

# 標準誤差
print("標準誤差を計算中...")
se_rcdc = compute_se(neg_ll_rcdc, est_rcdc)

print("=" * 60)
print("モデル3: 消費者属性なしランダム係数ロジット")
print("=" * 60)
print(f"対数尤度: {ll_rcdc:.6f}")
print(f"計算時間: {elapsed_rcdc:.1f}秒")
print(f"収束: {'成功' if result_rcdc.success else '失敗'}")
print()

coef_names_rcdc = ['price', 'Kinoko', 'Takenoko',
                    'sd.price', 'sd.Kinoko', 'sd.Takenoko']
for i, name in enumerate(coef_names_rcdc):
    se_val = se_rcdc[i] if se_rcdc is not None else np.nan
    print(f"  {name:14s}: est={est_rcdc[i]:10.6f}, se={se_val:.4f}")


## 10. モデル4: 消費者属性を入れたランダム係数ロジット（スクラッチ推定）

**注意**: この推定には数分〜十数分かかることがあります。

In [ ]:
# 初期値（Rと同じ）
ini_rcdc_atr = np.array([0.6, 20, 20, 0.1, 1, 1,
                          0, 3, 3, 0, 0, 0, 0, -3, -3, 0, 0, 1, 1, 1, 1], dtype=float)

def neg_ll_rcdc_atr(param):
    return -f_likelihood_rcdclogit_with_atr(param, respondent_choices, respondent_atrs,
                                             tau, times_draw, n_respondents)

print("ランダム係数ロジット（属性あり）の推定を開始...")
start_time = time.time()
result_rcdc_atr = minimize(neg_ll_rcdc_atr, ini_rcdc_atr, method='BFGS',
                            options={'maxiter': 10000, 'gtol': 1e-5})
elapsed_rcdc_atr = time.time() - start_time

est_rcdc_atr = result_rcdc_atr.x
ll_rcdc_atr = -result_rcdc_atr.fun

# 標準誤差
print("標準誤差を計算中...")
se_rcdc_atr = compute_se(neg_ll_rcdc_atr, est_rcdc_atr)

print("=" * 60)
print("モデル4: 消費者属性ありランダム係数ロジット")
print("=" * 60)
print(f"対数尤度: {ll_rcdc_atr:.6f}")
print(f"計算時間: {elapsed_rcdc_atr:.1f}秒")
print(f"収束: {'成功' if result_rcdc_atr.success else '失敗'}")
print()

coef_names_rcdc_atr = [
    'price', 'Kinoko', 'Takenoko',
    'sd.price', 'sd.Kinoko', 'sd.Takenoko',
    'price:genderfemale', 'Takenoko:genderfemale', 'Kinoko:genderfemale',
    'price:familyhouse', 'Takenoko:familyhouse', 'Kinoko:familyhouse',
    'price:adult', 'Takenoko:adult', 'Kinoko:adult',
    'price:regionKansai', 'price:regionOversea',
    'Takenoko:regionKansai', 'Takenoko:regionOversea',
    'Kinoko:regionKansai', 'Kinoko:regionOversea'
]

for i, name in enumerate(coef_names_rcdc_atr):
    se_val = se_rcdc_atr[i] if se_rcdc_atr is not None else np.nan
    print(f"  {name:28s}: est={est_rcdc_atr[i]:10.6f}, se={se_val:.4f}")


## 11. 推定結果のまとめ

In [ ]:
# 全モデルの結果をCSVにまとめる（Rの result_coef_scratch.csv に対応）

# モデル1の結果 (price, Kinoko, Takenoko)
names_1 = ['price', 'Kinoko', 'Takenoko']
est_1 = est_ml[[2, 0, 1]]
se_1 = se_ml[[2, 0, 1]] if se_ml is not None else np.array([np.nan]*3)

# モデル2の結果
est_2 = est_ml_atr
se_2 = se_ml_atr if se_ml_atr is not None else np.full(18, np.nan)

# モデル3の結果
est_3 = est_rcdc
se_3 = se_rcdc if se_rcdc is not None else np.full(6, np.nan)

# モデル4の結果
est_4 = est_rcdc_atr
se_4 = se_rcdc_atr if se_rcdc_atr is not None else np.full(21, np.nan)

# DataFrameを構築（Rと同じフォーマット）
all_coef_names = coef_names_rcdc_atr  # 最も多いモデル4の係数名をベースに

rows = []
for i, name in enumerate(all_coef_names):
    row = {'coefname': name}
    # モデル1
    if name in names_1:
        j = names_1.index(name)
        row['est_vec'] = est_1[j]
        row['se_vec'] = se_1[j]
    # モデル2
    if i < 18:
        row['est_vec_ml_atr'] = est_2[i]
        row['se_vec_ml_atr'] = se_2[i]
    # モデル3
    if i < 6:
        row['est_vec_rcdc'] = est_3[i]
        row['se_vec_rcdc'] = se_3[i]
    # モデル4
    row['est_vec_rcdc_with_atr'] = est_4[i]
    row['se_vec_with_atr'] = se_4[i]
    rows.append(row)

dt_result = pd.DataFrame(rows)
dt_result.to_csv(OUTPUT_DIR / "result_coef_scratch_python.csv", index=False)
print("推定結果を result_coef_scratch_python.csv に保存しました")
print()

# 表形式で表示
print("=" * 90)
print("推定結果サマリー")
print("=" * 90)
print(f"{'':28s} {'MNL':>12s} {'MNL+atr':>12s} {'RCDC':>12s} {'RCDC+atr':>12s}")
print("-" * 90)
for i, name in enumerate(all_coef_names):
    vals = []
    if name in names_1:
        j = names_1.index(name)
        vals.append(f"{est_1[j]:10.4f}")
    else:
        vals.append(f"{'':>10s}")
    if i < 18:
        vals.append(f"{est_2[i]:10.4f}")
    else:
        vals.append(f"{'':>10s}")
    if i < 6:
        vals.append(f"{est_3[i]:10.4f}")
    else:
        vals.append(f"{'':>10s}")
    vals.append(f"{est_4[i]:10.4f}")
    print(f"  {name:26s} {'  '.join(vals)}")

print("-" * 90)
print(f"  {'対数尤度':26s} {ll_ml:10.2f}  {ll_ml_atr:10.2f}  {ll_rcdc:10.2f}  {ll_rcdc_atr:10.2f}")
print("=" * 90)


## 12. 消費者属性を考慮した選好パラメータおよびWTPの分布

モデル4（属性ありランダム係数ロジット）の推定結果を用いて、
各個人の選好パラメータの分布とWTPの分布を計算します。

In [ ]:
# データ準備: 全ダミー変数にする（Rの data_for_param に対応）
data_for_param = KT_2024.copy()
data_for_param['gender_d'] = data_for_param['gender'].map({"1 : 男性": 0, "2 : 女性": 1})
data_for_param['adult_d'] = (data_for_param['age'] >= 20).astype(int)
data_for_param['kansai_d'] = data_for_param['region'].map({
    "1 : 北海道地方": 0, "2 : 東北地方": 0, "3 : 関東地方": 0,
    "4 : 中部地方": 0, "5 : 近畿地方": 1, "6 : 中国地方": 1,
    "7 : 四国地方": 1, "8 : 九州地方(沖縄含む)": 1, "9 : 海外": 0
})
data_for_param['oversea_d'] = data_for_param['region'].map({
    "1 : 北海道地方": 0, "2 : 東北地方": 0, "3 : 関東地方": 0,
    "4 : 中部地方": 0, "5 : 近畿地方": 0, "6 : 中国地方": 0,
    "7 : 四国地方": 0, "8 : 九州地方(沖縄含む)": 0, "9 : 海外": 1
})

# モデル4の推定値から分布のパラメータを取得
# est_rcdc_atr の構造:
# [0] beta, [1] alpha_kinoko, [2] alpha_takenoko,
# [3] sigma_price, [4] sigma_kinoko, [5] sigma_takenoko,
# [6] price:gender, [7] Takenoko:gender, [8] Kinoko:gender,
# [9] price:familyhouse, [10] Takenoko:familyhouse, [11] Kinoko:familyhouse,
# [12] price:adult, [13] Takenoko:adult, [14] Kinoko:adult,
# [15] price:regionKansai, [16] price:regionOversea,
# [17] Takenoko:regionKansai, [18] Takenoko:regionOversea,
# [19] Kinoko:regionKansai, [20] Kinoko:regionOversea

mean_kinoko = est_rcdc_atr[1]
mean_takenoko = est_rcdc_atr[2]
mean_price = est_rcdc_atr[0]
sigma_kinoko_est = est_rcdc_atr[4]
sigma_takenoko_est = est_rcdc_atr[5]
sigma_price_est = est_rcdc_atr[3]

# 各個人の属性パラメータの平均を計算
gender_arr = data_for_param['gender_d'].values
fh_arr = data_for_param['familyhouse'].values
adult_arr = data_for_param['adult_d'].values
kansai_arr = data_for_param['kansai_d'].values
oversea_arr = data_for_param['oversea_d'].values

# Rと同じ係数インデックスでパラメータを計算
# param_Takenoko = mean + gender*coef[5-1] + familyhouse*coef[8-1] + adult*coef[11-1] + kansai*coef[15-1] + oversea*coef[16-1]
# Rは1-indexed, ここでは0-indexedで対応
# R: coefficients[5] = Takenoko:gender → est_rcdc_atr[7]
# R: coefficients[8] = Takenoko:familyhouse → est_rcdc_atr[10]
# R: coefficients[11] = Takenoko:adult → est_rcdc_atr[13]
# R: coefficients[15] = Takenoko:kansai → est_rcdc_atr[17]
# R: coefficients[16] = Takenoko:oversea → est_rcdc_atr[18]

param_Takenoko = (mean_takenoko
                  + gender_arr * est_rcdc_atr[7]
                  + fh_arr * est_rcdc_atr[10]
                  + adult_arr * est_rcdc_atr[13]
                  + kansai_arr * est_rcdc_atr[17]
                  + oversea_arr * est_rcdc_atr[18])

param_Kinoko = (mean_kinoko
                + gender_arr * est_rcdc_atr[8]
                + fh_arr * est_rcdc_atr[11]
                + adult_arr * est_rcdc_atr[14]
                + kansai_arr * est_rcdc_atr[19]
                + oversea_arr * est_rcdc_atr[20])

param_price = (mean_price
               + gender_arr * est_rcdc_atr[6]
               + fh_arr * est_rcdc_atr[9]
               + adult_arr * est_rcdc_atr[12]
               + kansai_arr * est_rcdc_atr[15]
               + oversea_arr * est_rcdc_atr[16])

print(f"回答者数: {len(param_Kinoko)}")
print(f"param_Kinoko: mean={param_Kinoko.mean():.4f}, sd={param_Kinoko.std():.4f}")
print(f"param_Takenoko: mean={param_Takenoko.mean():.4f}, sd={param_Takenoko.std():.4f}")
print(f"param_price: mean={param_price.mean():.6f}, sd={param_price.std():.6f}")


In [ ]:
# 各個人について1000回ドローして、計 N*1000 個の個人選好パラメータのドローを得る
n_draws = 1000
n_persons = len(param_Kinoko)
total_draws = n_persons * n_draws

alpha_Kinoko_vec = np.zeros(total_draws)
alpha_Takenoko_vec = np.zeros(total_draws)
beta_vec = np.zeros(total_draws)

for i in range(n_persons):
    rng = np.random.RandomState(100 + i + 1)  # Rの set.seed(100+i) に対応
    draws = rng.randn(n_draws)
    alpha_Kinoko_vec[n_draws * i : n_draws * (i + 1)] = sigma_kinoko_est * draws + param_Kinoko[i]

    draws = rng.randn(n_draws)
    alpha_Takenoko_vec[n_draws * i : n_draws * (i + 1)] = sigma_takenoko_est * draws + param_Takenoko[i]

    draws = rng.randn(n_draws)
    beta_vec[n_draws * i : n_draws * (i + 1)] = sigma_price_est * draws + param_price[i]

print(f"総ドロー数: {total_draws}")
print(f"alpha_Kinoko: mean={alpha_Kinoko_vec.mean():.4f}")
print(f"alpha_Takenoko: mean={alpha_Takenoko_vec.mean():.4f}")
print(f"beta: mean={beta_vec.mean():.6f}")


### 12.1 きのこ vs たけのこの選好の分布

In [ ]:
# きのことたけのこの係数の差の分布（ヒストグラム）
fig, ax = plt.subplots(figsize=(8, 6))
diff = alpha_Kinoko_vec - alpha_Takenoko_vec
ax.hist(diff, bins=np.arange(diff.min(), diff.max() + 0.75, 0.75),
        edgecolor='white', alpha=0.7)
ax.set_xlabel("Kinoko - Takenoko")
ax.set_ylabel("frequency")
ax.set_title("Preference Kinoko over Takenoko")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig2_1_LHS_hist_preference_KT.pdf")
plt.show()


In [ ]:
# 密度プロット
fig, ax = plt.subplots(figsize=(8, 6))
sns.kdeplot(diff, ax=ax)
ax.set_xlabel("Kinoko - Takenoko")
ax.set_ylabel("density")
ax.set_title("Preference Kinoko over Takenoko")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig_dist_preference_KT.pdf")
plt.show()


### 12.2 価格係数の分布

In [ ]:
# 価格係数の分布（ヒストグラム）
fig, ax = plt.subplots(figsize=(8, 6))
ax.hist(beta_vec, bins=np.arange(beta_vec.min(), beta_vec.max() + 0.0075, 0.0075),
        edgecolor='white', alpha=0.7)
ax.set_xlabel("beta")
ax.set_ylabel("frequency")
ax.set_title("Beta (price coefficient)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig2_1_RHS_hist_price_coef.pdf")
plt.show()


In [ ]:
# 密度プロット
fig, ax = plt.subplots(figsize=(8, 6))
sns.kdeplot(beta_vec, ax=ax)
ax.set_xlabel("beta")
ax.set_ylabel("density")
ax.set_title("Beta (price coefficient)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig_dist_price_coef.pdf")
plt.show()


### 12.3 WTPの分布

In [ ]:
# WTPの計算
brand_Kinoko = alpha_Kinoko_vec / beta_vec
brand_Takenoko = alpha_Takenoko_vec / beta_vec

# WTPの分布
fig, ax = plt.subplots(figsize=(12, 6))
sns.kdeplot(brand_Kinoko, ax=ax, label='WTP_Kinoko')
sns.kdeplot(brand_Takenoko, ax=ax, label='WTP_Takenoko')
ax.set_xlim(0, 500)
ax.set_xlabel("WTP")
ax.set_ylabel("density")
ax.legend()
ax.set_title("Distribution of WTP")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig2_2_dist_wtp.pdf")
plt.show()


## 13. 需要曲線と収入曲線

たけのこの価格を200円に固定し、きのこの価格を100〜250円で動かした場合の需要と収入を計算します。

In [ ]:
# たけのこの価格を200円で固定
price_Takenoko_fixed = 200

# きのこの価格を100円から250円まで5円ずつ動かす
price_vec = np.arange(100, 255, 5)
kinoko_demand_vec = np.zeros(len(price_vec))

for i, p in enumerate(price_vec):
    result = f_demand(alpha_Kinoko_vec, alpha_Takenoko_vec, beta_vec, p, price_Takenoko_fixed)
    kinoko_demand_vec[i] = result[0]

data_demand = pd.DataFrame({
    'price': price_vec,
    'demand': kinoko_demand_vec,
    'revenue': price_vec * kinoko_demand_vec
})

print("需要曲線の計算完了")
data_demand.head()


In [ ]:
# 需要曲線の描画
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(data_demand['demand'], data_demand['price'])
ax.set_xlabel("Demand of Kinoko")
ax.set_ylabel("Price of Kinoko in JPY")
ax.set_title("Demand Curve of Kinoko when Takenoko's price = JPY 200")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "demand_Kinoko.pdf")
plt.show()


In [ ]:
# シェアベースの需要曲線
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(data_demand['demand'] / total_draws, data_demand['price'])
ax.set_xlabel("Share of Kinoko")
ax.set_ylabel("Price of Kinoko in JPY")
ax.set_title("Demand Curve (Share) of Kinoko when Takenoko's price = JPY 200")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "share_kinoko.pdf")
plt.show()


In [ ]:
# 収入曲線の描画
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(data_demand['price'], data_demand['revenue'] / 10000)
ax.set_xlabel("Price of Kinoko in JPY")
ax.set_ylabel("Revenue of Kinoko in 10000 JPY")
ax.set_title("Revenue Curve of Kinoko when Takenoko's price = JPY 200")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "Revenue_Kinoko.pdf")
plt.show()


## 14. Rの推定結果との比較検証

Rのスクラッチ推定結果（`result_coef_scratch.csv`）およびパッケージ推定結果（`result.txt`）と比較します。

In [ ]:
# Rのスクラッチ推定結果を読み込み
r_results = pd.read_csv(DATA_DIR.parent / "output" / "result_coef_scratch.csv")

# Rのパッケージ推定結果（result.txt から手動抽出した対数尤度）
r_ll = {
    'MNL': -1067.662,
    'RCDC': -727.742,
    'MNL+atr': -1038.798,
    'RCDC+atr': -713.796
}

print("=" * 80)
print("R vs Python 推定結果の比較")
print("=" * 80)

# === モデル1: MNL ===
print("\n--- モデル1: 消費者属性なし多項ロジット ---")
print(f"{'係数名':20s} {'R':>12s} {'Python':>12s} {'差':>12s} {'相対誤差(%)':>12s}")
r_ml_coefs = {'price': 0.0574460508, 'Kinoko': 11.6514737226, 'Takenoko': 12.2031093665}
py_ml_coefs = {'price': est_ml[2], 'Kinoko': est_ml[0], 'Takenoko': est_ml[1]}
for name in ['price', 'Kinoko', 'Takenoko']:
    r_val = r_ml_coefs[name]
    py_val = py_ml_coefs[name]
    diff = py_val - r_val
    rel = abs(diff / r_val) * 100 if r_val != 0 else np.nan
    print(f"  {name:18s} {r_val:12.6f} {py_val:12.6f} {diff:12.6f} {rel:10.4f}%")
print(f"  {'対数尤度':18s} {r_ll['MNL']:12.3f} {ll_ml:12.3f} {ll_ml - r_ll['MNL']:12.3f}")

# === モデル2: MNL+atr ===
print("\n--- モデル2: 消費者属性あり多項ロジット ---")
print(f"{'係数名':28s} {'R':>12s} {'Python':>12s} {'差':>12s}")
r_ml_atr_vals = r_results[r_results['est_vec_ml_atr'].notna()][['coefname', 'est_vec_ml_atr']]
for _, row in r_ml_atr_vals.iterrows():
    name = row['coefname']
    r_val = row['est_vec_ml_atr']
    idx = coef_names_ml_atr.index(name) if name in coef_names_ml_atr else -1
    py_val = est_ml_atr[idx] if idx >= 0 else np.nan
    diff = py_val - r_val if not np.isnan(py_val) else np.nan
    print(f"  {name:26s} {r_val:12.6f} {py_val:12.6f} {diff:12.6f}")
print(f"  {'対数尤度':26s} {r_ll['MNL+atr']:12.3f} {ll_ml_atr:12.3f} {ll_ml_atr - r_ll['MNL+atr']:12.3f}")

# === モデル3: RCDC ===
print("\n--- モデル3: 消費者属性なしランダム係数ロジット ---")
print(f"{'係数名':20s} {'R':>12s} {'Python':>12s} {'差':>12s}")
r_rcdc_vals = r_results[r_results['est_vec_rcdc'].notna()][['coefname', 'est_vec_rcdc']]
for _, row in r_rcdc_vals.iterrows():
    name = row['coefname']
    r_val = row['est_vec_rcdc']
    idx = coef_names_rcdc.index(name) if name in coef_names_rcdc else -1
    py_val = est_rcdc[idx] if idx >= 0 else np.nan
    diff = py_val - r_val if not np.isnan(py_val) else np.nan
    print(f"  {name:18s} {r_val:12.6f} {py_val:12.6f} {diff:12.6f}")
print(f"  {'対数尤度':18s} {r_ll['RCDC']:12.3f} {ll_rcdc:12.3f} {ll_rcdc - r_ll['RCDC']:12.3f}")

# === モデル4: RCDC+atr ===
print("\n--- モデル4: 消費者属性ありランダム係数ロジット ---")
print(f"{'係数名':28s} {'R':>12s} {'Python':>12s} {'差':>12s}")
r_rcdc_atr_vals = r_results[['coefname', 'est_vec_rcdc_with_atr']].dropna()
for _, row in r_rcdc_atr_vals.iterrows():
    name = row['coefname']
    r_val = row['est_vec_rcdc_with_atr']
    idx = coef_names_rcdc_atr.index(name) if name in coef_names_rcdc_atr else -1
    py_val = est_rcdc_atr[idx] if idx >= 0 else np.nan
    diff = py_val - r_val if not np.isnan(py_val) else np.nan
    print(f"  {name:26s} {r_val:12.6f} {py_val:12.6f} {diff:12.6f}")
print(f"  {'対数尤度':26s} {r_ll['RCDC+atr']:12.3f} {ll_rcdc_atr:12.3f} {ll_rcdc_atr - r_ll['RCDC+atr']:12.3f}")

# === 総合判定 ===
print("\n" + "=" * 80)
print("総合判定")
print("=" * 80)
# MNLの係数差をチェック
ml_max_diff = max(abs(py_ml_coefs[k] - r_ml_coefs[k]) for k in r_ml_coefs)
ml_ll_diff = abs(ll_ml - r_ll['MNL'])
ml_ok = ml_max_diff < 0.01 and ml_ll_diff < 0.1
print(f"  MNL: 最大係数差={ml_max_diff:.6f}, 対数尤度差={ml_ll_diff:.3f} → {'OK' if ml_ok else 'NG'}")

ml_atr_ll_diff = abs(ll_ml_atr - r_ll['MNL+atr'])
ml_atr_ok = ml_atr_ll_diff < 1.0
print(f"  MNL+atr: 対数尤度差={ml_atr_ll_diff:.3f} → {'OK' if ml_atr_ok else 'NG (数値最適化の差による可能性あり)'}")

rcdc_ll_diff = abs(ll_rcdc - r_ll['RCDC'])
rcdc_ok = rcdc_ll_diff < 5.0  # RCDCはシミュレーションの乱数差で多少ずれる
print(f"  RCDC: 対数尤度差={rcdc_ll_diff:.3f} → {'OK' if rcdc_ok else 'NG (乱数生成の差による可能性あり)'}")

rcdc_atr_ll_diff = abs(ll_rcdc_atr - r_ll['RCDC+atr'])
rcdc_atr_ok = rcdc_atr_ll_diff < 10.0
print(f"  RCDC+atr: 対数尤度差={rcdc_atr_ll_diff:.3f} → {'OK' if rcdc_atr_ok else 'NG (乱数生成の差による可能性あり)'}")

print()
if ml_ok:
    print("MNLモデル: Rの結果とほぼ一致しています。")
print("注意: RCDCモデルではRとPythonの乱数生成器が異なるため、")
print("      シミュレーションに基づくドローが完全には一致しません。")
print("      対数尤度や係数に多少の差異が生じるのは正常です。")


## 完了

全ての推定と可視化が完了しました。出力ファイルは `output/` ディレクトリに保存されています。